<div style="padding:10px;
            color:#E55604;
            margin:10px;
            font-size:130%;
            display:fill;
            border-radius:5px;
            border-style: solid;
            border-color: #FF9B50;
            background-color:#000000;
            overflow:hidden;
            font-weight:400"><b>Importing Libraries</b>
</div>

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


# 1. Load and Prepare Data


In [14]:
url = "../data/heart_disease.csv"

df = pd.read_csv(url)

df = df.replace("?", np.nan)
df = df.dropna()
df = df.astype(float)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]
y = (y > 0).astype(int)  # Binary labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Baseline Random Forest


In [15]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, rf.predict(X_test))
print("Baseline RF Accuracy:", baseline_acc)

Baseline RF Accuracy: 0.8


# 3. GridSearchCV for Random Forest


In [16]:
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("\nBest RF Params (GridSearch):", grid_rf.best_params_)
print("Best RF Score (CV):", grid_rf.best_score_)



Best RF Params (GridSearch): {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
Best RF Score (CV): 0.85177304964539


# 4. RandomizedSearchCV for SVM


In [17]:
param_dist_svm = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly"]
}

rand_svm = RandomizedSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_distributions=param_dist_svm,
    n_iter=10,  # number of random combinations
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

rand_svm.fit(X_train_scaled, y_train)

print("\nBest SVM Params (RandomizedSearch):", rand_svm.best_params_)
print("Best SVM Score (CV):", rand_svm.best_score_)



Best SVM Params (RandomizedSearch): {'kernel': 'rbf', 'gamma': 0.01, 'C': 1}
Best SVM Score (CV): 0.8390070921985815


# 5. Evaluate Optimized Models


In [19]:
best_rf = grid_rf.best_estimator_
best_svm = rand_svm.best_estimator_

models = {
    "Optimized RF": best_rf,
    "Optimized SVM": best_svm
}

for name, model in models.items():
    if "SVM" in name:
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:,1]
    else:
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"\n{name} Results:")
    print(f" Accuracy: {acc:.4f}")
    print(f" Precision: {prec:.4f}")
    print(f" Recall: {rec:.4f}")
    print(f" F1-score: {f1:.4f}")
    print(f" AUC: {auc:.4f}")


Optimized RF Results:
 Accuracy: 0.8000
 Precision: 0.8077
 Recall: 0.7500
 F1-score: 0.7778
 AUC: 0.9230

Optimized SVM Results:
 Accuracy: 0.8667
 Precision: 0.9167
 Recall: 0.7857
 F1-score: 0.8462
 AUC: 0.9364


# Model Export

In [22]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Best model (Optimized SVM from GridSearchCV)
best_svm = SVC(C=10, gamma=0.01, kernel='rbf', probability=True, random_state=42)

# Create pipeline (scaling + SVM)
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', best_svm)
])

# Fit on the full dataset
svm_pipeline.fit(X, y)

# Save as pickle file
joblib.dump(svm_pipeline, "../models/heart_disease_model.pkl")

print("✅ Optimized SVM model saved as best_svm_model.pkl")


✅ Optimized SVM model saved as best_svm_model.pkl
